In [ ]:
import os
import re
import pandas as pd

out_dir = 'Goodreads_data'

In [ ]:
ids = []

for file in os.listdir(out_dir):
    if re.search(r'tsv$',file):
        fh = open(os.path.join(out_dir,file))
        for line in fh:
            parts = re.split(r'\t',line)
            if re.search(r'^\d+$',parts[0]):
                ids.append(parts[0])
            
ids = list(set(ids))
print(len(ids))

In [ ]:
path = '/opt/homebrew/bin/chromedriver'

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

from selenium.webdriver.chrome.service import Service

service = Service(executable_path=path)
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)

In [ ]:
from gr_credentials import *
print(username)

In [ ]:
login_url = 'https://www.goodreads.com/ap/signin?language=en_US&openid.assoc_handle=amzn_goodreads_web_na&openid.claimed_id=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.identity=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.mode=checkid_setup&openid.ns=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0&openid.pape.max_auth_age=0&openid.return_to=https%3A%2F%2Fwww.goodreads.com%2Fap-handler%2Fsign-in'

#print(login_url)
driver.get(login_url)
time.sleep(2)


field = driver.find_element(by=By.ID,value='ap_email')
field.send_keys(username)
time.sleep(0.5)

field = driver.find_element(by=By.ID,value='ap_password')
field.send_keys(password)

button = driver.find_element(by=By.ID,value='signInSubmit')
button.click()

In [ ]:
for review_id in ids:

    file_path = f'Goodreads_reviews/{review_id}.txt'
    if not os.path.exists(file_path):
        out = open(file_path,'w',encoding='utf-8')
        
        try:

            time.sleep(1.5)
            driver.get(f"https://www.goodreads.com/review/show/{review_id}?utm_campaign=reviews&utm_medium=widget")
            try: 
                field = driver.find_element(by=By.CLASS_NAME,value='reviewText')
                review_text = field.text
                review_text = re.sub(r'\s+',' ',review_text)
            except:
                review_text = ''

            date_published = driver.find_element(by=By.XPATH, value="//span[@itemprop='datePublished']")

            rating  = driver.find_element(by=By.XPATH, value="//meta[@itemprop='ratingValue']")
            
            try:
                review_likes = driver.find_element(by=By.XPATH, value="//span[@class='likesCount']")
                if review_likes is not None:
                    nr_likes = review_likes.text
                    nr_likes = re.sub(r'\D+','',nr_likes)
                    nr_likes = int(nr_likes)
            except:
                nr_likes = 0


            out.write(f'{date_published.text}\n')
            out.write(f'{rating.get_attribute('content')}\n')
            out.write(f'{nr_likes}\n')
            out.write(f'{review_text}\n')
        except:
            print(f'Problem for {review_id}')
        out.close()
        

In [ ]:
time.sleep(3)
driver.close()